In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="adnankarim/urdu_asr_data", 
    repo_type="dataset", local_dir="./urdu_asr_data", allow_patterns="*/*.parquet")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 46 files: 100%|██████████| 46/46 [00:30<00:00,  1.52it/s]


'/home/ubuntu/urdu_asr_data'

In [3]:
files = glob('urdu_asr_data/*/*.parquet')
len(files)

46

In [4]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in files:
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in tqdm(range(len(df))):
            t = df['text'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}"
            })
        
    return data

In [5]:
data = multiprocessing(files, loop, cores = 20)

100%|██████████| 2135/2135 [02:17<00:00, 15.52it/s]


In [6]:
audio_files = [d['audio_filename'] for d in data]

with open('urdu_asr_data-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [7]:
with open('urdu_asr_data.json', 'w') as fopen:
    json.dump(data, fopen)

In [9]:
# !zip -rq urdu_asr_data.zip urdu_asr_data
# !hf upload malaysia-ai/Multilingual-TTS urdu_asr_data.zip --repo-type=dataset